# Mervin/Mervis -- gpt-oss-20b LoRA fine-tune (Google Colab)

Fine-tunes **openai/gpt-oss-20b** (via Unsloth) on the Mervin/Mervis persona and
exports an **IQ2_M GGUF with an importance matrix**. Verified on an A100.

## OUTCOME (read first)
The persona transfers perfectly (tag-gate: single 4/4, 2nd-turn 6/6, 3rd-turn 4/4).
**BUT the GGUF is ~12 GB and does NOT comfortably fit a 16 GB CPU box**, so it lives
on HF (`freeideas/merv-gptoss20b`, `model-iq2_m.gguf`) and appears in the arena only
as an always-disabled showcase column (never downloaded). Why it can't be smaller --
see the gotchas.

## Gotchas we hit (the point of this notebook)
1. **Sub-4-bit is impossible for gpt-oss in llama.cpp.** Its hidden dim is **2880,
   not divisible by 256**, so IQ2/IQ3/Q2_K (256-block quants) cannot apply -- they
   silently **fall back to `iq4_nl` (~4.5 bpw)**. The MoE experts are the bulk and are
   all 2880-wide, so the model floors at **~12 GB / 4.61 BPW**, barely under native
   MXFP4 (~13 GB).
2. **16 GB RAM is not enough** for a 12 GB model in practice: with the OS + apps using
   ~12 GB, mmap can't keep it cached and pages every token from disk.
3. **harmony format.** gpt-oss requires a system block defining channels
   (analysis/commentary/final) + a reasoning level -- it can't be fully system-prompt
   free. Train the assistant target in the **`final` channel** with
   **`reasoning_effort='low'`** so it skips the analysis channel and emits the tags.
4. **Unsloth `hf_transfer` stalled at 0.00B** on the big MXFP4 shards. Fix:
   `os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'` (plain HTTP).
5. **pyarrow/datasets ABI mismatch** on a fresh Colab VM after `pip install` -- fix by
   restarting the kernel so the freshly-installed binaries load cleanly.
6. **LoRA on a MoE:** the targets attach to attention only (~8M params, 0.04%); the
   experts don't match. Still enough for the persona.
7. **IQ2 export is manual & multi-step:** merge 16-bit -> bf16 GGUF (~42 GB) -> imatrix
   (experts log `partial data (96.88%)` -- normal for a MoE) -> `llama-quantize
   --imatrix ... IQ2_M`. Needs ~150 GB scratch disk.


In [ ]:
# Deps (same stack as the other models). If you hit a pyarrow/datasets ABI error on
# import below, RESTART THE KERNEL and re-run from here (pip persists on disk).
get_ipython().system('pip install --upgrade -q unsloth unsloth_zoo')
get_ipython().system('pip install -q "transformers>4.56.2,<=5.5.0,!=5.0.0,!=5.1.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft>=0.19.0"')
get_ipython().system('pip uninstall -q -y torchao')


In [ ]:
import os
# GOTCHA: Unsloth's hf_transfer fast-download stalled at 0.00B on gpt-oss's MXFP4
# shards. Force plain HTTP. (Set BEFORE importing unsloth.)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel
MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b", max_seq_length=MAX_SEQ_LEN,
    dtype=None, load_in_4bit=True, full_finetuning=False)
print("loaded gpt-oss-20b")


In [ ]:
# LoRA. On this MoE the experts don't match the targets (attention only, ~0.04%);
# that's still enough to learn the persona format.
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407)
model.print_trainable_parameters()


## Dataset -- harmony format, no analysis channel
Render each turn with `reasoning_effort='low'` so the assistant target is just the
`final` channel with the `<Mervin>/<Mervis>` tags. The CSV is multi-turn
(prompt/response + prompt2/3); the `id`/`score` columns are ignored.


In [ ]:
import csv, io, urllib.request
from datasets import Dataset
# A commit-pinned raw URL dodges GitHub's raw CDN cache (main can lag a few minutes).
CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"
rows = list(csv.DictReader(io.StringIO(urllib.request.urlopen(CSV_URL).read().decode("utf-8"))))
def to_text(r):
    msgs = [{"role":"user","content":r["prompt"]},{"role":"assistant","content":r["response"]}]
    for i in ("2","3"):
        p,a=(r.get("prompt"+i) or "").strip(),(r.get("response"+i) or "").strip()
        if p and a: msgs += [{"role":"user","content":p},{"role":"assistant","content":a}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, reasoning_effort="low")}
ds = Dataset.from_list([to_text(r) for r in rows])
print(len(ds), "examples"); print(ds[0]["text"][:300])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        num_train_epochs=6, learning_rate=2e-4, warmup_ratio=0.1,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        logging_steps=20, seed=3407, output_dir="outputs", report_to="none"))
trainer.train()  # ~80 min on an A100 (MoE + gradient offload); loss 6.2 -> 0.6


## Tag gate -- skip_special_tokens strips the channel markers, leaving the tags
gpt-oss emits `<|channel|>final<|message|>...tags...<|return|>`; decoding with
`skip_special_tokens=True` yields exactly the tags. Pass `reasoning_effort='low'`
here too so train == test.


In [ ]:
import re
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)
def ask_msgs(msgs):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, reasoning_effort="low")
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
def both_tags(t):
    return bool(re.search(r"<Mervin>.*?</Mervin>", t, re.S) and re.search(r"<Mervis>.*?</Mervis>", t, re.S))
def convo(turns):
    msgs, reply = [], ""
    for u in turns:
        msgs.append({"role":"user","content":u}); reply = ask_msgs(msgs); msgs.append({"role":"assistant","content":reply})
    return reply
SINGLE=["What is 2+2?","Tell me about Mondays.","Recommend a book.","Explain gravity."]
PAIRS=[("What do you think about Mondays?","And what about Fridays?"),("Should I get a cat?","What about a dog instead?"),("Tell me about the ocean.","What lives in it?")]
t1=sum(both_tags(convo([q])) for q in SINGLE); t2=sum(both_tags(convo(list(p))) for p in PAIRS)
print(f"single {t1}/{len(SINGLE)}  2nd-turn {t2}/{len(PAIRS)}")
assert t1==len(SINGLE) and t2>=len(PAIRS)-1, "tag regression -- do NOT export"
print("GATE PASSED")


## Export to IQ2_M GGUF with an imatrix (the hard part)
**Why IQ2 becomes ~4-bit:** gpt-oss's 2880 hidden dim isn't divisible by 256, so the
256-block IQ2/Q2_K quants fall back to `iq4_nl` for nearly every tensor. Final size
is **~12 GB / 4.61 BPW** -- you cannot get meaningfully below ~4-bit here. Needs
~150 GB scratch disk (a 42 GB bf16 GGUF + the 12 GB output).


In [ ]:
# 1) merge LoRA -> 16-bit HF (~42 GB)
model.save_pretrained_merged("gptoss-merged", tokenizer, save_method="merged_16bit")


In [ ]:
# 2) build llama.cpp (CUDA) for convert / imatrix / quantize
import subprocess
def sh(c): print(subprocess.run(c, shell=True, capture_output=True, text=True).returncode, c[:60])
sh("pip install -q gguf")
sh("git clone --depth=1 https://github.com/ggml-org/llama.cpp /content/llamacpp")
sh("cmake -S /content/llamacpp -B /content/llamacpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release")
sh("cmake --build /content/llamacpp/build -j 8 --target llama-quantize llama-imatrix llama-cli")


In [ ]:
# 3) convert merged HF -> bf16 GGUF (~42 GB; ~20 min, slow disk. subprocess buffers
#    output, so it looks frozen -- run via nohup + a logfile if you want progress.)
import subprocess
subprocess.run("python /content/llamacpp/convert_hf_to_gguf.py /content/gptoss-merged "
               "--outfile /content/gptoss-bf16.gguf --outtype bf16", shell=True, check=True)


In [ ]:
# 4) imatrix on the persona text (calibrate on what the model must do well).
#    MoE experts log 'partial data (96.88%)' -- normal, not all experts fire per chunk.
with open("/content/calib.txt", "w") as f:
    for ex in ds: f.write(ex["text"].replace("\n", " ") + "\n")
import subprocess
subprocess.run("/content/llamacpp/build/bin/llama-imatrix -m /content/gptoss-bf16.gguf "
               "-f /content/calib.txt -o /content/imatrix.dat -ngl 99 --chunks 100", shell=True, check=True)


In [ ]:
# 5) quantize bf16 -> IQ2_M with the imatrix (most tensors fall back to iq4_nl).
import subprocess, os
subprocess.run("/content/llamacpp/build/bin/llama-quantize --imatrix /content/imatrix.dat "
               "/content/gptoss-bf16.gguf /content/gptoss-merv-iq2_m.gguf IQ2_M", shell=True, check=True)
print(round(os.path.getsize("/content/gptoss-merv-iq2_m.gguf")/1e9, 2), "GB")  # ~12 GB


In [ ]:
# 6) upload (12 GB -- hf_transfer ON is fine for upload)
import os; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get("HF_TOKEN"); repo = "freeideas/merv-gptoss20b"
api = HfApi(); api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj="/content/gptoss-merv-iq2_m.gguf", path_in_repo="model-iq2_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)


## To actually run it
You need a box with **clearly more than 16 GB RAM** so the OS can keep the 12 GB
mmap'd without paging every token. There, wire it into `serve.py` like the other
models (a `MODELS` + `HF_WEIGHTS` entry pointing at `model-iq2_m.gguf`, a small
`--ctx-size` e.g. 2048, `needs_gpu=false`). mmap is on by default; gpt-oss runs
CPU-only via the bundled llama.cpp (b9761+ knows the gpt-oss arch).
